# Solar Filament Segmentation 2026: full pipeline

This notebook runs every step, from raw H-alpha images to the submission CSV:

1. Look at the data
2. Preprocessing (solar disk detection and limb-darkening correction)
3. Training (optional; the trained weights are in `weights/`)
4. Validation: PQ, Dice and IoU distributions, fragmentation
5. Visual check of predictions
6. Predict the test set and write `submission.csv`

Set `DATA` to the folder that contains `train/` and `test/`.

In [ ]:
# On Kaggle or Colab, first clone the repo and install the requirements, e.g.:
# !git clone https://github.com/<you>/<repo>.git && cd <repo> && pip install -r requirements.txt
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch

from filseg.data import DataPaths, find_data_root, group_by_stem, index_images, load_annotations, read_image
from filseg.preprocess import normalize

DATA = find_data_root(Path("data"))  # or e.g. Path("/kaggle/input/...")
paths = DataPaths(DATA)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DATA, DEVICE)

## 1. The data

In [ ]:
items = load_annotations(paths.train_json)
images = index_images(paths.train_images)
groups = group_by_stem(items)
n_fil = [len(i.filaments) for i in items]
print(f"{len(items)} annotation sets, {len(groups)} distinct images, {len(index_images(paths.test_images))} test images")
print(f"filaments per annotation set: mean {np.mean(n_fil):.1f}, max {max(n_fil)}")
print(f"images with >1 annotator: {sum(len(g) > 1 for g in groups.values())}")

In [ ]:
from filseg.viz import overlay
item = items[0]
img = read_image(images[item.stem])
plt.figure(figsize=(8, 8)); plt.imshow(overlay(img, gt_mask=item.union_mask())); plt.title(item.image_id); plt.axis("off");

## 2. Preprocessing
The disk is darker at the edge (limb darkening). Dividing by the radial brightness profile makes filaments a consistent negative contrast everywhere on the disk.

In [ ]:
contrast, disk = normalize(img)
fig, ax = plt.subplots(1, 2, figsize=(14, 7))
ax[0].imshow(img, cmap="gray"); ax[0].set_title("raw")
ax[1].imshow(contrast, cmap="gray", vmin=-4, vmax=2); ax[1].set_title("limb-darkening corrected")
for a in ax: a.axis("off")

## 3. Training (optional)
Training needs a GPU (about 1-2 hours on a Kaggle T4). Skip this cell to use the weights saved in `weights/`.

In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    !python -m filseg.train --data "{DATA}" --out weights/model.pt --epochs 40
    !python -m filseg.tune  --data "{DATA}" --weights weights/model.pt

## 4. Validation scores

In [ ]:
from filseg.dataset import FilamentDataset
from filseg.metrics import format_summary, summarize
from filseg.model import load_checkpoint
from filseg.postprocess import PostParams
from filseg.validate import predict_val_probs, score_probs

model, meta = load_checkpoint("weights/model.pt", DEVICE)
params = PostParams.load("weights/postprocess.json") if Path("weights/postprocess.json").exists() else PostParams()
val_items = [i for i in items if i.stem in set(meta["val_stems"])]
val_ds = FilamentDataset(val_items, images, meta["size"], None, False)
probs = predict_val_probs(model, val_ds, DEVICE)
scores = score_probs(probs, val_items, params)
print(params)
print(format_summary(summarize(scores)))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].hist([s.pq for s in scores], bins=20); ax[0].set_title("PQ per image")
ax[1].hist([s.dice for s in scores], bins=20); ax[1].set_title("Dice per image")
ax[2].hist([s.iou for s in scores], bins=20); ax[2].set_title("IoU per image")
plt.tight_layout()

## 5. Visual check
Coloured = predicted filaments (one colour each), white outline = ground truth.

In [ ]:
from filseg.infer import predict_instances
for item in val_items[:3]:
    img = read_image(images[item.stem])
    labels = predict_instances(model, img, meta["size"], DEVICE, params)
    plt.figure(figsize=(9, 9)); plt.imshow(overlay(img, labels, item.union_mask()))
    plt.title(f"{item.image_id}: {labels.max()} predicted, {len(item.filaments)} true"); plt.axis("off")

## 6. Test-set predictions -> submission.csv

In [ ]:
!python -m filseg.predict --images "{paths.test_images}" --weights weights/model.pt --out submission.csv